# Gap-Fill Retraining — Fixing Roman Urdu / Mid-Length Long-Query Recall

**Created 2026-08-10.** This notebook is the authoritative training pipeline for the
gap-filled model (`models/svm_classifier.pkl`, `models/scaler.pkl` as of this date).
Per the lesson learned from the notebook 06 vs 14 mismatch, this notebook actually
calls `joblib.dump()` to produce the deployed model — it is not a prototype.

## Root cause

The original 369-query training set had a hard gap in query length:
- **short** label: always 2-4 words (178 queries at exactly 2 words)
- **long** label: always 10-19 words

**Zero training examples existed in the 5-9 word range.** Real-world queries —
especially natural Roman Urdu phrasing, which tends to be more concise — commonly
fall in this untrained gap and were being misclassified as "short". This was
previously (imprecisely) framed as a pure "Roman Urdu weakness"; checking the
actual misclassified queries shows 3 of 13 were native Urdu-script queries in the
5-9 word range too. It is a **training-data distribution gap**, not a purely
script-specific issue — Roman Urdu is just disproportionately affected because
natural Roman Urdu queries skew shorter than the synthetic 10-19 word Urdu-script
"long" templates.

This also explains why the earlier dataset-expansion attempt (documented in
`validation_response.py::test3_dataset_expansion`) made things *worse* (70% vs 74%):
it added 18 new short (2-word) queries but only 2 new long queries, both 17-20 words
— it didn't touch the 5-9 word gap at all, and further biased the classifier toward
predicting "short".

## Fix

40 new training queries (20 Urdu-script, 20 Roman Urdu), all genuinely 5-9 words,
all labeled "long", covering topics distinct from the 50-query external validation
set (floods, school fees, PSL, air quality, etc.) so the fix isn't tailored to the
test set. Appended to `data/training_queries_real.py`.

In [1]:
import sys, os, json
sys.path.insert(0, '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from validation_response import extract_features, load_roman_dict, load_original_queries

roman_dict = load_roman_dict()
queries = load_original_queries()
print(f"Total training queries (with gap-fill): {len(queries)}")
print(f"Short: {sum(1 for q,l in queries if l=='short')}  Long: {sum(1 for q,l in queries if l=='long')}")

wl = sorted(set(len(q.split()) for q,l in queries))
print(f"Word-count values now present in training data: {wl}")


Total training queries (with gap-fill): 409
Short: 193  Long: 216
Word-count values now present in training data: [2, 3, 4, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


## Train with identical hyperparameters to the previously deployed model (RBF, C=1.0, gamma='scale')

In [2]:
X_raw = np.array([extract_features(q, roman_dict) for q, _ in queries])
y = np.array([l for _, l in queries])

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# 5-fold CV sanity check - confirm the gap-fill doesn't break training accuracy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for tr, te in skf.split(X, y):
    m = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
    m.fit(X[tr], y[tr])
    cv_scores.append(m.score(X[te], y[te]))
cv_scores = np.array(cv_scores)
print(f"5-fold CV accuracy: {cv_scores.mean():.2%} (std {cv_scores.std():.2%})")

# Fit final model on all data
model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
model.fit(X, y)
print("Final model fitted on all", len(queries), "queries.")


5-fold CV accuracy: 100.00% (std 0.00%)
Final model fitted on all 409 queries.


## Before/after comparison: deployed (pre-fix) model vs new gap-filled model, on the same 50-query external validation set (from notebook 14) plus a brand-new 16-query holdout set (topics never seen in training or the 50-query set).

In [3]:
external_queries = [
    ("پاکستان نے بھارت کو ورلڈ کپ فائنل میں شکست دی", "long"),
    ("کرکٹ", "short"),
    ("بابر اعظم نے سنچری اسکور کی", "long"),
    ("میچ", "short"),
    ("قومی ٹیم کے کھلاڑیوں کی فٹنس رپورٹ جاری", "long"),
    ("pakistan cricket world cup final match result", "long"),
    ("score", "short"),
    ("babar azam ne century kaise score ki aaj ke match mein", "long"),
    ("wicket", "short"),
    ("pakistan team ne india ko kis tarah se harayaa", "long"),
    ("وزیراعظم", "short"),
    ("حکومت نے نئے بجٹ کا اعلان کر دیا", "long"),
    ("الیکشن", "short"),
    ("قومی اسمبلی میں اپوزیشن نے تحریک عدم اعتماد پیش کی", "long"),
    ("سیاست", "short"),
    ("PM ne naya budget announce kiya aaj assembly mein", "long"),
    ("election", "short"),
    ("opposition ne government ke khilaf motion submit kiya", "long"),
    ("wazir", "short"),
    ("pakistan mein siyasi buhran ke baad nai hakumat bani", "long"),
    ("ڈالر", "short"),
    ("عالمی بینک نے پاکستان کو قرض دینے سے انکار کر دیا", "long"),
    ("مہنگائی", "short"),
    ("اسٹیٹ بینک نے شرح سود میں اضافہ کر دیا", "long"),
    ("بجٹ", "short"),
    ("dollar rate aaj kitna hai", "long"),
    ("mehngai", "short"),
    ("imf ne pakistan ko loan dene ki sharait rakhi hain", "long"),
    ("market", "short"),
    ("pakistan ki economy mein behteri aa rahi hai ya nahi", "long"),
    ("ہسپتال", "short"),
    ("کورونا وائرس کی نئی لہر نے پاکستان میں دستک دے دی", "long"),
    ("دوائی", "short"),
    ("وزارت صحت نے ویکسین مہم شروع کرنے کا فیصلہ کیا", "long"),
    ("علاج", "short"),
    ("hospital", "short"),
    ("corona ki nayi lehar pakistan mein phail rahi hai", "long"),
    ("dawai", "short"),
    ("sehat ka khayal kaise rakha jaye garmiyon mein", "long"),
    ("vaccine", "short"),
    ("موبائل", "short"),
    ("مصنوعی ذہانت نے طب کے شعبے میں انقلاب برپا کر دیا", "long"),
    ("انٹرنیٹ", "short"),
    ("پاکستان میں فائیو جی سروس کب شروع ہوگی", "long"),
    ("ٹیکنالوجی", "short"),
    ("mobile", "short"),
    ("AI ne medical field mein kitni taraqqi ki hai abhi tak", "long"),
    ("internet", "short"),
    ("pakistan mein 5G service kab tak launch hogi officially", "long"),
    ("tech", "short"),
]

fresh_holdout = [
    ("موسم رپورٹ", "short"), ("فلم ریلیز", "short"), ("عمرہ ویزا", "short"),
    ("weather report", "short"), ("movie release", "short"), ("umrah visa", "short"),
    ("اسلام آباد میں آج بارش کا امکان ہے", "long"),
    ("نئی فلم نے باکس آفس پر ریکارڈ بنایا", "long"),
    ("حج کے لیے درخواستیں کب شروع ہوں گی", "long"),
    ("قومی ہاکی ٹیم نے فائنل جیت لیا", "long"),
    ("kal barish hone ka imkan hai islamabad mein", "long"),
    ("nai film ne box office pe record banaya", "long"),
    ("hajj ke liye application kab shuru hogi", "long"),
    ("hockey team ne final match jeet liya", "long"),
    ("موسمیاتی تبدیلی کی وجہ سے شمالی علاقہ جات میں برف باری کا نظام متاثر ہو رہا ہے", "long"),
    ("entertainment industry mein naye directors ko kaam karne ke mauqe kam milte hain kyunki competition bohat zyada hai", "long"),
]

def evaluate(mdl, scl, data, label):
    qs = [q for q, _ in data]
    ys = np.array([l for _, l in data])
    Xr = np.array([extract_features(q, roman_dict) for q in qs])
    Xt = scl.transform(Xr)
    preds = mdl.predict(Xt)
    acc = (preds == ys).mean()
    lm, sm = ys == 'long', ys == 'short'
    lr = (preds[lm] == 'long').mean() if lm.sum() else float('nan')
    sr = (preds[sm] == 'short').mean() if sm.sum() else float('nan')
    print(f"{label:<45} Acc={acc:.2%}  LongRecall={lr:.2%}  ShortRecall={sr:.2%}")
    return acc, lr, sr

old_model = pickle.load(open('../models/svm_classifier_PRE_GAPFIX_backup.pkl', 'rb'))
old_scaler = pickle.load(open('../models/scaler_PRE_GAPFIX_backup.pkl', 'rb'))

print("=== 50-query external validation set ===")
evaluate(old_model, old_scaler, external_queries, "OLD model (pre-gap-fix, backed up)")
new_ext = evaluate(model, scaler, external_queries, "NEW gap-filled model")

print()
print("=== Fresh holdout: 16 brand-new queries, topics never seen anywhere ===")
evaluate(old_model, old_scaler, fresh_holdout, "OLD model (pre-gap-fix, backed up)")
new_fresh = evaluate(model, scaler, fresh_holdout, "NEW gap-filled model")


=== 50-query external validation set ===
OLD model (pre-gap-fix, backed up)            Acc=74.00%  LongRecall=45.83%  ShortRecall=100.00%
NEW gap-filled model                          Acc=98.00%  LongRecall=95.83%  ShortRecall=100.00%

=== Fresh holdout: 16 brand-new queries, topics never seen anywhere ===
OLD model (pre-gap-fix, backed up)            Acc=62.50%  LongRecall=40.00%  ShortRecall=100.00%
NEW gap-filled model                          Acc=100.00%  LongRecall=100.00%  ShortRecall=100.00%


## Save the new model

Old model/scaler backed up as `models/svm_classifier_PRE_GAPFIX_backup.pkl` and `models/scaler_PRE_GAPFIX_backup.pkl` before overwriting, in case this needs to be rolled back.

In [4]:
pickle.dump(model, open('../models/svm_classifier.pkl', 'wb'))
pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))
print("Saved: models/svm_classifier.pkl, models/scaler.pkl")

info = {
    "feature_order": ["urdu_ratio","roman_ratio","has_urdu","has_roman","query_len","char_len","mixed","urdu_chars"],
    "training_queries": len(queries),
    "kernel": "rbf", "C": 1.0, "gamma": "scale", "probability": True,
    "cv_accuracy_mean": float(cv_scores.mean()),
    "cv_accuracy_std": float(cv_scores.std()),
    "external_validation_50q_accuracy": float(new_ext[0]),
    "external_validation_50q_long_recall": float(new_ext[1]),
    "fresh_holdout_16q_accuracy": float(new_fresh[0]),
    "fresh_holdout_16q_long_recall": float(new_fresh[1]),
    "trained_on": "2026-08-10",
    "notes": "Gap-fill fix: added 40 queries (5-9 word range, both scripts) to close a training-data distribution gap that caused mid-length/Roman-Urdu long queries to be misclassified as short."
}
with open('../models/training_info.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print("Updated models/training_info.json")


Saved: models/svm_classifier.pkl, models/scaler.pkl
Updated models/training_info.json


## Honest caveats (not yet done — flagged, not hidden)

- **Leave-one-topic-out** was not re-run with the new gap-fill topics added as formal
  topic tags; the 100% LOTO result documented elsewhere is on the original 369-query
  topic set only.
- The fresh 16-query holdout, while genuinely unseen, is still author-created (not
  independently labeled) — same limitation as the original 50-query external set.
- This has **not yet been reviewed by the supervisor**. Treat the 98%/100% numbers as
  a validated candidate fix pending sign-off, not a final claim, in the thesis defense
  until Dr. Adnan Aslam has reviewed it.